<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

**Solução**:

In [20]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    X, y = fetch_openml("Fashion-MNIST", version=1, as_frame=False, return_X_y=True)

    y = y.astype(int)

    X = X / 255.0

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=10000, random_state=seed, stratify=y
    )

    return X_train, X_test, y_train, y_test

Para Random Forest não é necessário normalizar os dados pois o modelo não depende de escala, já para o AdaBoost é recomendado normalizar os dados, devido a sensibilidade do modelo a distribuição dos dados.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,      
        random_state=seed,     
        n_jobs=-1              
    )
    
    model.fit(X_train, y_train)
    
    return model


def train_adaboost(X_train, y_train, seed):
    model = AdaBoostClassifier(
        n_estimators=100,      
        random_state=seed      
    )
    
    model.fit(X_train, y_train)
    
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [22]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    return acc, report

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [23]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    
    return acc

O Random Forest utiliza árvores de decisão com max_depth=None por padrão, permitindo que cresçam até o máximo possível. Isso possibilita que cada árvore atinja 100% de acurácia no treino, pois consegue memorizar os dados. O overfitting começa a ocorrer quando a profundidade é muito alta, fazendo com que o modelo capture ruídos em vez de padrões gerais. Mas, no Random Forest esse efeito é reduzido devido ao uso de múltiplas árvores e amostragem aleatória.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [26]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

X_train, X_test, y_train, y_test = load_data(42)

# Random Forest
model_rf = train_random_forest(X_train, y_train, 42)
acc_rf, report_rf = evaluate(model_rf, X_test, y_test)

print("=== Random Forest ===")
print("Acurácia:", acc_rf)
print(report_rf)


# AdaBoost
model_ab = train_adaboost(X_train, y_train, 42)
acc_ab, report_ab = evaluate(model_ab, X_test, y_test)

print("\n=== AdaBoost ===")
print("Acurácia:", acc_ab)
print(report_ab)

=== Random Forest ===
Acurácia: 0.8845
              precision    recall  f1-score   support

           0       0.81      0.88      0.84      1000
           1       0.99      0.96      0.98      1000
           2       0.79      0.84      0.81      1000
           3       0.87      0.92      0.90      1000
           4       0.78      0.84      0.81      1000
           5       0.97      0.96      0.97      1000
           6       0.75      0.56      0.64      1000
           7       0.95      0.94      0.94      1000
           8       0.97      0.97      0.97      1000
           9       0.95      0.97      0.96      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


=== AdaBoost ===
Acurácia: 0.5654
              precision    recall  f1-score   support

           0       0.71      0.62      0.66      1000
           1       0.98      0.82      0.89      1000
    

O modelo Random Forest apresentou melhor desempenho inicial, com acurácia de aproximadamente 88%, além de maiores valores de precisão, recall e F1-score em comparação ao AdaBoost, que obteve cerca de 57% de acurácia. O Random Forest também apresentou desempenho mais equilibrado entre as classes, enquanto o AdaBoost apresentou dificuldades significativas, chegando a não reconhecer corretamente algumas classes (como a classe 7). Isso ocorre porque o Random Forest utiliza múltiplas árvores profundas e independentes, sendo mais robusto e capaz de capturar melhor padrões complexos, enquanto o AdaBoost, por padrão, utiliza modelos fracos, o que limita seu desempenho inicial em datasets mais desafiadores como o Fashion MNIST.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [27]:
# Random Forest
model_rf = train_random_forest(X_train, y_train, 42)
acc_rf, report_rf = evaluate(model_rf, X_test, y_test)

print("=== Random Forest ===")
print("Acurácia:", acc_rf)
print(report_rf)


# AdaBoost
model_ab = train_adaboost(X_train, y_train, 7)
acc_ab, report_ab = evaluate(model_ab, X_test, y_test)

print("\n=== AdaBoost ===")
print("Acurácia:", acc_ab)
print(report_ab)

=== Random Forest ===
Acurácia: 0.8845
              precision    recall  f1-score   support

           0       0.81      0.88      0.84      1000
           1       0.99      0.96      0.98      1000
           2       0.79      0.84      0.81      1000
           3       0.87      0.92      0.90      1000
           4       0.78      0.84      0.81      1000
           5       0.97      0.96      0.97      1000
           6       0.75      0.56      0.64      1000
           7       0.95      0.94      0.94      1000
           8       0.97      0.97      0.97      1000
           9       0.95      0.97      0.96      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


=== AdaBoost ===
Acurácia: 0.5654
              precision    recall  f1-score   support

           0       0.71      0.62      0.66      1000
           1       0.98      0.82      0.89      1000
    

Os resultados não apresentaram mudanças significativas mesmo ao utilizar seeds diferentes para os modelos. Isso indica que o experimento é relativamente estável. A reprodutibilidade é garantida pelo uso do parâmetro random_state, que controla a aleatoriedade interna dos algoritmos. Porém, para garantir uma comparação totalmente justa e reprodutível entre os modelos, o ideal seria utilizar o mesmo seed em todas as etapas do pipeline, incluindo carregamento dos dados e treinamento dos modelos. Ademais, a baixa variação nos resultados do AdaBoost sugere que o modelo possui alta estabilidade, mas também pode indicar limitação na sua capacidade de aprendizado para esse dataset.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
# Random Forest
model_rf = train_random_forest(X_train, y_train, 42)

# Acurácia no treino
train_acc_rf = accuracy_score(y_train, model_rf.predict(X_train))

# Acurácia no teste
test_acc_rf = accuracy_score(y_test, model_rf.predict(X_test))

print("Treino:", train_acc_rf)
print("Teste:", test_acc_rf)



Treino: 0.9999666666666667
Teste: 0.8845


In [29]:
# AdaBoost
model_ab = train_adaboost(X_train, y_train, 42)

# Acurácia no treino
train_acc_ab = accuracy_score(y_train, model_ab.predict(X_train))

# Acurácia no teste
test_acc_ab = accuracy_score(y_test, model_ab.predict(X_test))

print("Treino:", train_acc_ab)
print("Teste:", test_acc_ab)



Treino: 0.5720833333333334
Teste: 0.5654


Ao comparar a acurácia no conjunto de treino e teste, observa-se que o modelo Random Forest apresenta uma acurácia significativamente maior no treino (próxima de 100%) em comparação ao teste (cerca de 88%), indicando a presença de overfitting. Isso ocorre porque o modelo possui alta capacidade e consegue memorizar os dados de treino. Por outro lado, o AdaBoost apresenta baixa acurácia tanto no treino quanto no teste, sugerindo underfitting. Assim, o Random Forest tende a sofrer mais com overfitting, enquanto o AdaBoost, neste caso, sofre com baixa capacidade de aprendizado.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [15]:
# TODO: implemente load_data

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [16]:
# TODO: implemente load_data